# Oracle_DBA_Console

copyright 2026, Denis Rothman

📘 **User Guide: Oracle DBA & Cloud Control Center**

This notebook serves as the **Control Center** for your **Oracle 23ai Hybrid Vector Database**. It encompasses both the cloud infrastructure (hardware state) and table management for the entire repository, including Chapters 2 and 3.

It leverages Oracle 23ai's converged architecture to manage both standard Vector Stores (unstructured text) and Hybrid Schemas (tables combining relational data with vectors).

Follow the following workflow to manage your database lifecycle.

**1. OCI Infrastructure Control (Power & Billing)**

**Start & Wait**: Run the OCI initialization cells to power on your Autonomous Database instance. The script includes a "Waiter" that monitors the lifecycle state until it reaches AVAILABLE.

**Stop & Save**: At the end of your session, ensure you run the Teardown cell at the bottom of the notebook. This stops the instance and pauses per-second compute billing.

**2. Initialization & Connection**
Run the Installation and Connection steps to authenticate with your Oracle 23ai instance.

*Crucial Step*: Run the "Download Schema Registry" cell. This fetches the latest table definitions (create_script.py) from the repository, ensuring your schema matches the book's architecture.

**3. The Management Loop**
Use the Interactive Console at the bottom of the notebook to perform these actions:    

🔨 **CREATE**:
When to use: At the start of a new chapter or if you have dropped tables.     
What it does: Reads the downloaded registry and builds the table structure (DDL). It is safe to run multiple times; it will skip tables that already exist.   

📊 **VERIFY**:
When to use: Immediately after creation or before starting a chapter notebook.
What it does: Checks if tables exist and reports their row counts.   

📋 **DISPLAY**:
When to use: After running a chapter's ingestion notebook.
What it does: Fetches the first 10 rows. Use this to visually confirm that structured data (e.g., Salary, ID), text chunks, and vectors were inserted correctly.    

🧹 **CLEAN (Truncate)**:
When to use: If you want to restart ingestion without deleting the table structure.
What it does: Instantly removes all data but keeps the tables ready for new inserts.    

🔥 **DROP**:
When to use: To completely remove the schema (e.g., to save storage or reset environment).     
What it does: Permanently deletes the tables and their structure.

⚠️ **Note on Data Ingestion**:    
This console does not generate data. After you CREATE the tables here, switch to the specific Chapter Notebook to run the ingestion process (which vectorizes text and inserts hybrid records). You can then return here to DISPLAY the results.

⚠️**Run this notebook cell by cell** to monitor each step of the DBA Console Process.

# 1.Installation and Setup

## DBA Parameters



## 1.1 Global Configuration Flags

* **`unzip_wallet`**: Set to `True` only for the initial setup to extract the Oracle Wallet configuration. Once extracted, set to `False` to avoid redundant file operations.


In [ ]:
unzip_wallet=False  # True to unzip the wallet. False to only unzip the wallet once

## 1.2. Oracle environment Setup & Wallet Extraction

This step prepares the runtime environment by connecting to Google Drive and ensuring the Oracle Wallet is available. The Oracle Wallet contains the SSL/TLS certificates and configuration files (tnsnames.ora, sqlnet.ora) necessary for a secure connection to the Oracle Autonomous Database.

Google Drive Mount: Maps your personal Drive to /content/drive, allowing the notebook to read credentials and configuration files stored externally.

Wallet Unzipping: If unzip_wallet is set to True, the script searches for the wallet ZIP file in the specified path and extracts it. This ensures the connection drivers can locate the required security certificates.

Path Definition: Sets wallet_path to the directory where the unzipped configuration files reside, which will be passed to the Oracle driver in subsequent steps.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
if unzip_wallet==True:
  !unzip -o "/content/drive/MyDrive/files/oracle/Wallet_*.zip" -d "/content/drive/MyDrive/files/oracle"

## 1.3. Install Oracle Database Driver and Oracle Cloud Infrastructure (OCI)

Integrating the OCI (Oracle Cloud Infrastructure) SDK into your notebook is a smart way to automate cost savings.
This command installs the oracledb Python driver, which is the renamed and modernized successor to the legacy cx_Oracle driver. It serves as the bridge between this Python notebook and the Oracle Database.

Version Pinning (==3.4.1): The version is explicitly fixed to 3.4.1 to ensure stability and reproducibility. In production DBA scripts, pinning versions prevents unexpected updates or API changes from breaking the automation workflow.

Thin Client Mode: By default, this driver operates in "Thin" mode, meaning it communicates directly with the database using pure Python code without requiring local Oracle Client libraries (Instant Client).

The notebook integrates the **OCI (Oracle Cloud Infrastructure)** SDK to manage cost saving actions.

In [ ]:
!pip install oracledb==3.4.1

In [ ]:
import oracledb
print(oracledb.__version__)

In [ ]:
!pip install oci==2.167.0

## 1.4. Additional imports

In [ ]:
# Imports for this notebook
import json
import time
from tqdm.auto import tqdm
import tiktoken                                                         # tokenization
from tenacity import retry, stop_after_attempt, wait_random_exponential # to retry functions
import re
import textwrap
from IPython.display import display, Markdown
import copy

# 1.5 Starting the database

In [ ]:
from google.colab import userdata

# Pulling the ID from the hidden Colab Secrets (Safe for GitHub!)
adb_id = userdata.get('ADB_ID')

# Pulling the Oracle DSN from the hidden Colab Secrets (Safe for GitHub!)
oracle_dsn = userdata.get('oracle_dsn')

In [ ]:
import oci
import time

# Load your OCI config (ensure this file is on your Google Drive)
config = oci.config.from_file("/content/drive/MyDrive/files/oracle/oci_config")
client = oci.database.DatabaseClient(config)

def start_and_wait(db_client, db_id):
    db = db_client.get_autonomous_database(db_id).data
    if db.lifecycle_state == "STOPPED":
        print("Database is currently stopped. Starting now...")
        db_client.start_autonomous_database(db_id)

    # Waiter loop: Checks status every 15 seconds
    while True:
        db = db_client.get_autonomous_database(db_id).data
        status = db.lifecycle_state
        print(f"Current Status: {status}")
        if status == "AVAILABLE":
            print("Database is ready for connection.")
            break
        time.sleep(15)

start_and_wait(client, adb_id)

## 1.6.Establish Secure Database Connection

Wait for the database to start before attempting to connect to it.

This step handles the authentication and connection to the Oracle 23ai instance. It is designed to adhere to security best practices by strictly separating code from credentials.

External Credential Management: Instead of hardcoding sensitive passwords directly into the notebook (which is a security risk), the script reads a credentials.txt file stored securely on Google Drive.

Credential Parsing: The read_key_value_file helper function parses the external file to retrieve the username, password, Wallet password, and DSN (Data Source Name).

Connection Initialization: The oracledb.connect() method establishes the session using the extracted credentials and the local Wallet configuration path.

Connectivity Test: A simple "Heartbeat" query (SELECT ... FROM DUAL) is executed immediately to verify that the connection is active and stable before proceeding.

In [ ]:
import os

creds_path = "/content/drive/MyDrive/files/oracle/credentials.txt"
if not os.path.exists(creds_path):
    raise FileNotFoundError(f"Credentials file not found: {creds_path}")

def read_key_value_file(path):
    creds = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            k, v = line.split("=", 1)
            creds[k.strip()] = v.strip()
    return creds

creds = read_key_value_file(creds_path)

# Use values (do not print them)
user = creds.get("user")
password = creds.get("password")
wallet_password = creds.get("wallet_password")
dsn = creds.get("dsn", oracle_dsn)
wallet_path = creds.get("wallet_path", "/content/drive/MyDrive/files/oracle")

import oracledb
connection = oracledb.connect(
    user=user,
    password=password,
    dsn=dsn,
    config_dir=wallet_path,
    wallet_location=wallet_path,
    wallet_password=wallet_password
)

cursor = connection.cursor()
cursor.execute("SELECT 'Connected!' FROM dual")
print(cursor.fetchone())

# I.Schema Registry for Oracle DBA Console

In [ ]:
#@title download Schema registry
!curl -L -H "Accept: application/vnd.github.v3.raw" "https://api.github.com/repos/Denis2054/RAG-Driven-Generative-AI-2nd-Edition/contents/commons/create_script.py" -o create_script.py

# II.Table Registry

Selecting a chapter below for management operations like cleaning or dropping tables only affect the tables relevant to that specific chapter, preventing accidental data loss in other parts of your database environment.

In [ ]:
# ... existing code ...
# --- Configuration: Select Your Chapter Scope ---
# Update this variable to match the chapter you are currently working on.
CURRENT_SCOPE = 'CHAPTER_11'

# --- Table Registry ---
# A dictionary mapping chapters to their specific table architectures.
TABLE_REGISTRY = {
    'CHAPTER_2': [
        'CONTEXT_LIBRARY',
        'KNOWLEDGE_STORE'
    ],
    'CHAPTER_3': [
        'CONTEXT_LIBRARY',     # Shared infrastructure
        'KNOWLEDGE_STORE',     # Shared infrastructure
        'CANDIDATE_POOL',      # CH3 Specific: Hybrid Table (SQL + Vector)
        'RECRUITMENT_RULES'    # CH3 Specific: Agent Personas
    ],
    'CHAPTER_9': [
        'MEDIA_ASSETS',        # Video Catalog
        'MEDIA_SEGMENTS'       # Vectorized Video Segments
    ],
    'CHAPTER_10': [
        'CANDIDATE_POOL_GEO',  # Spatial Table
        'EMPLOYEES',           # Graph Vertex
        'REFERRALS'            # Graph Edge
    ],
    'CHAPTER_11': [
        'WORKLOAD_BENCHMARK'   # Scaling Benchmark Table
    ]
}

# III.Schema Operations Engine


In [ ]:
def manage_schema(cursor, mode='CLEAN'):
    """
    Iterates through the active_tables list to perform maintenance.

    Parameters:
    - cursor: The active Oracle database cursor.
    - mode: 'CLEAN' (default) to truncate, or 'DROP' to delete the table structure.
    """
    print(f"\n--- ⚙️ Starting Schema Operation: {mode} ---")
    print(f"Scope: {CURRENT_SCOPE}")

    # Iterate dynamically through the list defined in Step 1
    for table in active_tables:
        try:
            # 1. Idempotency Check: Verify table exists before touching it
            # This prevents errors if you try to drop a table that is already gone.
            cursor.execute(f"SELECT count(*) FROM user_tables WHERE table_name = '{table}'")
            exists = cursor.fetchone()[0]

            if exists:
                if mode == 'DROP':
                    print(f"   🔥 Dropping table: {table}...")
                    cursor.execute(f"DROP TABLE {table} PURGE")
                elif mode == 'CLEAN':
                    print(f"   🧹 Cleaning (Truncating) table: {table}...")
                    cursor.execute(f"TRUNCATE TABLE {table}")
            else:
                print(f"   ⚠️  Skipping {table}: Not found in database.")

        except Exception as e:
            print(f"   ❌ Error processing {table}: {e}")

    print("--- Operation Complete ---\n")

# IV.Schema Status Verification

It is critical to verify the state of the database. This function acts as an auditor. It iterates through the `active_tables` list dynamically and reports.

In [ ]:
def verify_schema_status(cursor):
    """
    Audits the current state of the active tables defined in the scope.
    """
    print(f"\n--- 📊 Schema Verification Report: {CURRENT_SCOPE} ---")

    for table in active_tables:
        try:
            # Check if table exists in Oracle metadata
            cursor.execute(f"SELECT count(*) FROM user_tables WHERE table_name = '{table}'")
            exists = cursor.fetchone()[0]

            if exists:
                # If it exists, count the rows
                cursor.execute(f"SELECT count(*) FROM {table}")
                count = cursor.fetchone()[0]

                # Visual feedback based on data presence
                status_icon = "✅ [EMPTY]" if count == 0 else "📦 [DATA PRESENT]"
                print(f"   Table: {table:<25} | Rows: {count:<5} | Status: {status_icon}")
            else:
                # If it doesn't exist
                print(f"   Table: {table:<25} | Status: ❌ [DROPPED / NOT FOUND]")

        except Exception as e:
            print(f"   ⚠️ Error auditing {table}: {e}")

    print("--- End of Report ---\n")

# V Oracle DBA Console

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import importlib
import sys

# --- 1. Configuration: Table Registry ---
# We add CHAPTERS here so the Console knows what to Verify/Drop/Display
TABLE_REGISTRY = {
    'CHAPTER_2': ['CONTEXT_LIBRARY', 'KNOWLEDGE_STORE'],
    'CHAPTER_3': ['CONTEXT_LIBRARY', 'KNOWLEDGE_STORE', 'CANDIDATE_POOL', 'RECRUITMENT_RULES'],
    'CHAPTER_9': ['MEDIA_ASSETS', 'MEDIA_SEGMENTS'],
    'CHAPTER_10': ['CANDIDATE_POOL_GEO', 'EMPLOYEES', 'REFERRALS'],
    'CHAPTER_11': ['WORKLOAD_BENCHMARK']
}

# --- 2. Define Logic Functions ---
def get_active_tables(scope):
    return TABLE_REGISTRY.get(scope, [])

def verify_schema(cursor, scope):
    tables = get_active_tables(scope)
    print(f"\n--- 📊 Schema Verification: {scope} ---")
    if not tables:
        print("❌ No tables defined for this scope.")
        return

    for table in tables:
        try:
            # Check existence
            cursor.execute(f"SELECT count(*) FROM user_tables WHERE table_name = '{table}'")
            exists = cursor.fetchone()[0]
            if exists:
                # Check row count
                cursor.execute(f"SELECT count(*) FROM {table}")
                count = cursor.fetchone()[0]
                status = "✅ [EMPTY]" if count == 0 else "📦 [DATA PRESENT]"
                print(f"   {table:<25} | Rows: {count:<5} | {status}")
            else:
                print(f"   {table:<25} | ❌ [NOT FOUND]")
        except Exception as e:
            print(f"   ⚠️ Error checking {table}: {e}")
    print("--- End of Report ---\n")

def display_table_data(cursor, scope):
    tables = get_active_tables(scope)
    print(f"\n--- 📋 Data Preview (First 10 Records): {scope} ---")

    for table in tables:
        try:
            # Check existence first
            cursor.execute(f"SELECT count(*) FROM user_tables WHERE table_name = '{table}'")
            exists = cursor.fetchone()[0]

            if exists:
                print(f"\n🔹 Table: {table}")
                sql = f"SELECT * FROM {table} FETCH FIRST 10 ROWS ONLY"
                cursor.execute(sql)

                columns = [col[0] for col in cursor.description]
                rows = cursor.fetchall()

                if rows:
                    df = pd.DataFrame(rows, columns=columns)
                    display(df)
                else:
                    print("   [Table is empty]")
            else:
                print(f"   {table:<25} | ❌ [NOT FOUND]")

        except Exception as e:
            print(f"   ⚠️ Error reading {table}: {e}")
    print("\n--- End of Preview ---")

def manage_schema(cursor, scope, mode):
    print(f"\n--- ⚙️ Executing {mode} on {scope} ---")

    # --- SPECIAL LOGIC FOR CREATE (Uses External Registry) ---
    if mode == 'CREATE':
        try:
            # 1. Dynamic Import/Reload of the Registry
            import create_script             # <--- IMPORT HAPPENS HERE
            importlib.reload(create_script)  # <--- RELOAD HAPPENS HERE
            # 1. Dynamic Import/Reload of the Registry
            import create_script
            importlib.reload(create_script)

            # 2. Get the catalog for this chapter
            catalog = create_script.DDL_CATALOG.get(scope)
            if not catalog:
                print(f"❌ No DDL definitions found in create_script.py for {scope}")
                return

            # 3. Iterate and Create
            for entry in catalog:
                table_name = entry['table_name']
                description = entry['description']
                sql = entry['sql']

                # Check if exists
                cursor.execute(f"SELECT count(*) FROM user_tables WHERE table_name = '{table_name}'")
                exists = cursor.fetchone()[0]

                if not exists:
                    print(f"   🔨 Creating {table_name}...")
                    print(f"      📝 Desc: {description}")
                    cursor.execute(sql)
                    print("      ✅ Created.")
                else:
                    print(f"   ⚠️  Skipping {table_name}: Already exists.")

            connection.commit()

        except ImportError:
            print("❌ Error: 'create_script.py' not found. Please run the download cell first.")
        except Exception as e:
            print(f"❌ Error during creation: {e}")
        return

    # --- LOGIC FOR DROP / CLEAN (Uses Standard List) ---
    tables = get_active_tables(scope)
    for table in tables:
        try:
            cursor.execute(f"SELECT count(*) FROM user_tables WHERE table_name = '{table}'")
            exists = cursor.fetchone()[0]

            if exists:
                if mode == 'DROP':
                    print(f"   🔥 Dropping: {table}...")
                    cursor.execute(f"DROP TABLE {table} PURGE")
                elif mode == 'CLEAN':
                    print(f"   fw Cleaning: {table}...")
                    cursor.execute(f"TRUNCATE TABLE {table}")
            else:
                print(f"   ⚠️ Skipping {table}: Not found.")
        except Exception as e:
            print(f"   ❌ Error processing {table}: {e}")

# --- 3. Build UI ---
style = {'description_width': 'initial'}

# Add CHAPTER labels to the dropdown options
scope_dropdown = widgets.Dropdown(
    options=['CHAPTER_2', 'CHAPTER_3', 'CHAPTER_9','CHAPTER_10','CHAPTER_11'],
    value='CHAPTER_3', # Default to the new chapter
    description='🎯 Scope:',
    style=style
)

action_dropdown = widgets.Dropdown(
    options=['VERIFY', 'DISPLAY', 'CREATE', 'CLEAN', 'DROP'],
    value='VERIFY',
    description='⚡ Action:',
    style=style
)

run_button = widgets.Button(
    description='Run Operation',
    button_style='primary',
    icon='play'
)

out = widgets.Output()

def on_button_clicked(b):
    with out:
        clear_output()
        scope = scope_dropdown.value
        action = action_dropdown.value

        if 'cursor' not in globals():
            print("❌ Error: Database cursor not found. Please connect to Oracle first.")
            return

        # Route to the correct function based on action
        if action == 'VERIFY':
            verify_schema(cursor, scope)
        elif action == 'DISPLAY':
            display_table_data(cursor, scope)
        else:
            manage_schema(cursor, scope, action)
            verify_schema(cursor, scope) # Always auto-verify after maintenance

run_button.on_click(on_button_clicked)

# --- 4. Display Interface ---
print("⬇️ Select your Chapter and Action, then click Run.")
display(widgets.HBox([scope_dropdown, action_dropdown, run_button]))
display(out)

# VI.Stopping the database

In [ ]:
# --- FINAL STEP: SESSION TEARDOWN & COST SAVING ---
close=False
if close==True:
  # 1. Close the Database Connection (The "Software" side)
  try:
      if 'connection' in locals() and connection:
          cursor.close()
          connection.close()
          print("✅ Oracle Database connection closed safely.")
  except Exception as e:
      print(f"⚠️ Connection was already closed or error occurred: {e}")

  # 2. Stop the Autonomous Database Instance (The "Hardware/Billing" side)
  print(f"🛑 Sending STOP command to: {adb_id}")

  try:
      # Use the OCI client created at the start of the notebook
      response = client.stop_autonomous_database(adb_id)
      print("⏳ Stop command accepted. Status:", response.data.lifecycle_state)
      print("💰 Compute billing has been paused. See you next session!")
  except Exception as e:
      print(f"❌ OCI Error: Could not stop the database. Please check the Cloud Console. Error: {e}")